In [2]:
!git clone https://github.com/davelee-uestc/nsf_debiasing.git
%cd nsf_debiasing
!pip install timm transformers scikit-learn tqdm wilds -q

fatal: destination path 'nsf_debiasing' already exists and is not an empty directory.
/content/nsf_debiasing/nsf_debiasing


In [3]:
# Fix mmcv
with open('/content/nsf_debiasing/ssc_common.py', 'r') as f:
    content = f.read()
content = content.replace('import mmcv', 'from tqdm import tqdm')
content = content.replace('prog=mmcv.ProgressBar(len(loader))', 'prog=tqdm(total=len(loader))')
content = content.replace('prog=mmcv.ProgressBar(N)', 'prog=tqdm(total=N)')
with open('/content/nsf_debiasing/ssc_common.py', 'w') as f:
    f.write(content)

# Fix wandb
with open('/content/nsf_debiasing/train_supervised.py', 'r') as f:
    content = f.read()
content = content.replace(
    'try:\n    import wandb\n    has_wandb = True\nexcept ImportError:\n    has_wandb = False',
    'has_wandb = False  # wandb disabled'
)
with open('/content/nsf_debiasing/train_supervised.py', 'w') as f:
    f.write(content)

print("✅ All patches applied!")

✅ All patches applied!


In [4]:
from google.colab import drive
drive.mount('/content/drive')

# Restore previous checkpoints if you saved them
import os
if os.path.exists('/content/drive/MyDrive/nsf_results/logs'):
    !cp -r /content/drive/MyDrive/nsf_results/logs /content/nsf_debiasing/
    print("✅ Previous checkpoints restored!")
else:
    print("No previous checkpoints found, starting fresh.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
No previous checkpoints found, starting fresh.


In [5]:
import json, os

# Paste your NEW token here after expiring the old one
kaggle_creds = {
    "username": "lakavatumeshchandra",
    "key": "KGAT_e8c1a7bd5c64ad23364044ee8403db16"
}

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)

os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("✅ Kaggle configured!")

✅ Kaggle configured!


In [6]:
!kaggle datasets download -d jessicali9530/celeba-dataset
print("✅ Downloaded!")

Dataset URL: https://www.kaggle.com/datasets/jessicali9530/celeba-dataset
License(s): other
celeba-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
✅ Downloaded!


In [7]:
import os
os.makedirs('/content/nsf_debiasing/celeba', exist_ok=True)
!unzip -q celeba-dataset.zip -d /content/nsf_debiasing/celeba/
!ls /content/nsf_debiasing/celeba/

replace /content/nsf_debiasing/celeba/list_attr_celeba.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/nsf_debiasing/celeba/list_bbox_celeba.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/nsf_debiasing/celeba/list_eval_partition.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/nsf_debiasing/celeba/list_landmarks_align_celeba.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
img_align_celeba      list_bbox_celeba.csv     list_landmarks_align_celeba.csv
list_attr_celeba.csv  list_eval_partition.csv  metadata.csv


In [8]:
import pandas as pd

attr = pd.read_csv('/content/nsf_debiasing/celeba/list_attr_celeba.csv')
print(attr.head())
print(attr.columns.tolist())

     image_id  5_o_Clock_Shadow  Arched_Eyebrows  Attractive  Bags_Under_Eyes  \
0  000001.jpg                -1                1           1               -1   
1  000002.jpg                -1               -1          -1                1   
2  000003.jpg                -1               -1          -1               -1   
3  000004.jpg                -1               -1           1               -1   
4  000005.jpg                -1                1           1               -1   

   Bald  Bangs  Big_Lips  Big_Nose  Black_Hair  ...  Sideburns  Smiling  \
0    -1     -1        -1        -1          -1  ...         -1        1   
1    -1     -1        -1         1          -1  ...         -1        1   
2    -1     -1         1        -1          -1  ...         -1       -1   
3    -1     -1        -1        -1          -1  ...         -1       -1   
4    -1     -1         1        -1          -1  ...         -1       -1   

   Straight_Hair  Wavy_Hair  Wearing_Earrings  Wearing_Hat  We

In [9]:
import pandas as pd
import numpy as np

# Load attributes and partition files
attr = pd.read_csv('/content/nsf_debiasing/celeba/list_attr_celeba.csv')
partition = pd.read_csv('/content/nsf_debiasing/celeba/list_eval_partition.csv')

# Merge on image_id
df = attr.merge(partition, on='image_id')

# Paper uses:
# y = Blond_Hair (1 = blonde, 0 = not blonde)
# spurious = Male (1 = male, 0 = female)
# split: 0=train, 1=val, 2=test

# Convert -1/1 to 0/1
df['y'] = (df['Blond_Hair'] == 1).astype(int)
df['spurious'] = (df['Male'] == 1).astype(int)
df['split'] = df['partition']

# img_filename needs to include subfolder
df['img_filename'] = 'img_align_celeba/' + df['image_id']

# Build final metadata
metadata = df[['img_filename', 'y', 'spurious', 'split']].copy()

# Save
metadata.to_csv('/content/nsf_debiasing/celeba/metadata.csv', index=False)

print("✅ metadata.csv created!")
print("Shape:", metadata.shape)
print(metadata.head())
print("\nSplit counts:", metadata['split'].value_counts().to_dict())
print("Label counts:", metadata['y'].value_counts().to_dict())
print("Spurious counts:", metadata['spurious'].value_counts().to_dict())

✅ metadata.csv created!
Shape: (202599, 4)
                  img_filename  y  spurious  split
0  img_align_celeba/000001.jpg  0         0      0
1  img_align_celeba/000002.jpg  0         0      0
2  img_align_celeba/000003.jpg  0         1      0
3  img_align_celeba/000004.jpg  0         0      0
4  img_align_celeba/000005.jpg  0         0      0

Split counts: {0: 162770, 2: 19962, 1: 19867}
Label counts: {0: 172616, 1: 29983}
Spurious counts: {0: 118165, 1: 84434}


In [10]:
!CUDA_VISIBLE_DEVICES=0 python3 train_supervised.py \
    --output_dir=/content/nsf_debiasing/logs/celeba/erm_seed1 \
    --num_epochs=7 \
    --eval_freq=1 \
    --save_freq=5 \
    --seed=1 \
    --weight_decay=1e-4 \
    --batch_size=100 \
    --init_lr=3e-3 \
    --scheduler=cosine_lr_scheduler \
    --data_dir=/content/nsf_debiasing/celeba \
    --data_transform=AugWaterbirdsCelebATransform \
    --dataset=SpuriousCorrelationDataset \
    --model=imagenet_resnet50_pretrained

2026-03-28 08:53:30.681760: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774688010.703891    9950 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774688010.711231    9950 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774688010.730322    9950 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774688010.730365    9950 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774688010.730370    9950 computation_placer.cc:177] computation placer alr

In [14]:
!ls /content/nsf_debiasing/celeba/
!ls /content/nsf_debiasing/celeba/img_align_celeba/ | head -5

img_align_celeba      list_bbox_celeba.csv     list_landmarks_align_celeba.csv
list_attr_celeba.csv  list_eval_partition.csv  metadata.csv
img_align_celeba


In [15]:
!ls /content/nsf_debiasing/celeba/img_align_celeba/ | wc -l
!ls /content/nsf_debiasing/celeba/img_align_celeba/ | head -5

1
img_align_celeba


In [16]:
!ls /content/nsf_debiasing/celeba/img_align_celeba/img_align_celeba/ | head -5
!ls /content/nsf_debiasing/celeba/img_align_celeba/img_align_celeba/ | wc -l

000001.jpg
000002.jpg
000003.jpg
000004.jpg
000005.jpg
202599


In [17]:
!CUDA_VISIBLE_DEVICES=0 python3 train_supervised.py \
    --output_dir=/content/nsf_debiasing/logs/celeba/erm_seed1 \
    --num_epochs=5 \
    --eval_freq=1 \
    --save_freq=5 \
    --seed=1 \
    --weight_decay=1e-4 \
    --batch_size=100 \
    --init_lr=3e-3 \
    --scheduler=cosine_lr_scheduler \
    --data_dir=/content/nsf_debiasing/celeba \
    --data_transform=AugWaterbirdsCelebATransform \
    --dataset=SpuriousCorrelationDataset \
    --model=imagenet_resnet50_pretrained

2026-03-28 08:40:55.476620: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774687255.512280    6358 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774687255.523546    6358 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774687255.551598    6358 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774687255.551635    6358 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774687255.551643    6358 computation_placer.cc:177] computation placer alr

In [18]:
import pandas as pd

df = pd.read_csv('/content/nsf_debiasing/celeba/metadata.csv')
print("Metadata sample filenames:")
print(df['img_filename'].head())

# Check what actually exists
import os
actual = os.listdir('/content/nsf_debiasing/celeba/img_align_celeba/')
actual.sort()
print("\nActual files sample:")
print(actual[:5])

Metadata sample filenames:
0    img_align_celeba/000001.jpg
1    img_align_celeba/000002.jpg
2    img_align_celeba/000003.jpg
3    img_align_celeba/000004.jpg
4    img_align_celeba/000005.jpg
Name: img_filename, dtype: object

Actual files sample:
['img_align_celeba']


In [19]:
import os, shutil

src = '/content/nsf_debiasing/celeba/img_align_celeba/img_align_celeba'
dst = '/content/nsf_debiasing/celeba/img_align_celeba'

# Move all files from nested folder to parent
!mv /content/nsf_debiasing/celeba/img_align_celeba/img_align_celeba/* \
    /content/nsf_debiasing/celeba/img_align_celeba/

# Remove empty nested folder
!rm -rf /content/nsf_debiasing/celeba/img_align_celeba/img_align_celeba

print("✅ Done!")

/bin/bash: line 1: /usr/bin/mv: Argument list too long
✅ Done!


In [21]:
import os, shutil
from tqdm import tqdm

src = '/content/nsf_debiasing/celeba/img_align_celeba/img_align_celeba'
dst = '/content/nsf_debiasing/celeba/img_align_celeba'

files = os.listdir(src)
print(f"Moving {len(files)} files...")

for f in tqdm(files):
    shutil.move(os.path.join(src, f), os.path.join(dst, f))

# Remove empty folder
os.rmdir(src)
print("✅ Done!")

FileNotFoundError: [Errno 2] No such file or directory: '/content/nsf_debiasing/celeba/img_align_celeba/img_align_celeba'

In [22]:
import os
files = os.listdir('/content/nsf_debiasing/celeba/img_align_celeba/')
files.sort()
print("Total files:", len(files))
print("First 5:", files[:5])
print("000001.jpg exists:", os.path.exists('/content/nsf_debiasing/celeba/img_align_celeba/000001.jpg'))


Total files: 0
First 5: []
000001.jpg exists: False


In [23]:
!ls /content/*.zip

ls: cannot access '/content/*.zip': No such file or directory


In [24]:
import json, os
os.makedirs('/root/.kaggle', exist_ok=True)

kaggle_creds = {
    "username": "lakavatumeshchandra",  # ← paste yours
    "key": "KGAT_7baa497cda1de9bd450e47ee27a4b218"            # ← paste new key (remember to expire old one!)
}
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!kaggle datasets download -d jessicali9530/celeba-dataset
print("✅ Downloaded!")

Dataset URL: https://www.kaggle.com/datasets/jessicali9530/celeba-dataset
License(s): other
celeba-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
✅ Downloaded!


In [27]:
import zipfile, os
from tqdm import tqdm

dst = '/content/nsf_debiasing/celeba/img_align_celeba'
os.makedirs(dst, exist_ok=True)

print("Extracting images...")
with zipfile.ZipFile('/content/celeba-dataset.zip', 'r') as z:
    image_files = [f for f in z.namelist() if f.endswith('.jpg')]
    print(f"Found {len(image_files)} images")

    for f in tqdm(image_files):
        filename = os.path.basename(f)
        with z.open(f) as src, open(os.path.join(dst, filename), 'wb') as out:
            out.write(src.read())

print("✅ Done!")
print("Total files:", len(os.listdir(dst)))

Extracting images...


FileNotFoundError: [Errno 2] No such file or directory: '/content/celeba-dataset.zip'

In [11]:
!find / -name "celeba-dataset.zip" 2>/dev/null

/content/nsf_debiasing/nsf_debiasing/celeba-dataset.zip


In [12]:
import zipfile, os
from tqdm import tqdm

dst = '/content/nsf_debiasing/celeba/img_align_celeba'
os.makedirs(dst, exist_ok=True)

zip_path = '/content/nsf_debiasing/nsf_debiasing/celeba-dataset.zip'  # ← correct path

with zipfile.ZipFile(zip_path, 'r') as z:
    image_files = [f for f in z.namelist() if f.endswith('.jpg')]
    print(f"Found {len(image_files)} images, extracting...")
    for f in tqdm(image_files):
        filename = os.path.basename(f)
        with z.open(f) as src, open(os.path.join(dst, filename), 'wb') as out:
            out.write(src.read())

print("✅ Done!", len(os.listdir(dst)), "files")

Found 202599 images, extracting...


100%|██████████| 202599/202599 [00:32<00:00, 6253.94it/s]


✅ Done! 202600 files


In [13]:
import pandas as pd

attr = pd.read_csv('/content/nsf_debiasing/celeba/list_attr_celeba.csv')
partition = pd.read_csv('/content/nsf_debiasing/celeba/list_eval_partition.csv')
df = attr.merge(partition, on='image_id')
df['y'] = (df['Blond_Hair'] == 1).astype(int)
df['spurious'] = (df['Male'] == 1).astype(int)
df['split'] = df['partition']
df['img_filename'] = 'img_align_celeba/' + df['image_id']
df[['img_filename','y','spurious','split']].to_csv(
    '/content/nsf_debiasing/celeba/metadata.csv', index=False)
print("✅ metadata ready!")

✅ metadata ready!


In [14]:
!CUDA_VISIBLE_DEVICES=0 python3 train_supervised.py \
    --output_dir=/content/nsf_debiasing/logs/celeba/erm_seed1 \
    --num_epochs=5 \
    --eval_freq=1 \
    --save_freq=1 \
    --seed=1 \
    --weight_decay=1e-4 \
    --batch_size=100 \
    --init_lr=3e-3 \
    --scheduler=cosine_lr_scheduler \
    --data_dir=/content/nsf_debiasing/celeba \
    --data_transform=AugWaterbirdsCelebATransform \
    --dataset=SpuriousCorrelationDataset \
    --model=imagenet_resnet50_pretrained

2026-03-28 09:03:47.657547: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774688627.680186   12705 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774688627.687399   12705 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774688627.707549   12705 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774688627.707599   12705 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774688627.707604   12705 computation_placer.cc:177] computation placer alr

In [15]:
import shutil
shutil.copytree(
    '/content/nsf_debiasing/logs/celeba/erm_seed1',
    '/content/drive/MyDrive/nsf_results/celeba_erm_seed1',
    dirs_exist_ok=True
)
print("✅ Saved to Drive!")

✅ Saved to Drive!


In [16]:
!python3 ssc.py celeba

2026-03-28 12:08:37.888560: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774699718.040036   62031 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774699718.082063   62031 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774699718.383844   62031 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774699718.383886   62031 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774699718.383891   62031 computation_placer.cc:177] computation placer alr

In [17]:
import os
os.chdir('/content/nsf_debiasing')

# Fix mmcv
with open('ssc_common.py', 'r') as f:
    content = f.read()
content = content.replace('import mmcv', 'from tqdm import tqdm')
content = content.replace('prog=mmcv.ProgressBar(len(loader))', 'prog=tqdm(total=len(loader))')
content = content.replace('prog=mmcv.ProgressBar(N)', 'prog=tqdm(total=N)')
with open('ssc_common.py', 'w') as f:
    f.write(content)

# Fix wandb
with open('train_supervised.py', 'r') as f:
    content = f.read()
content = content.replace(
    'try:\n    import wandb\n    has_wandb = True\nexcept ImportError:\n    has_wandb = False',
    'has_wandb = False  # wandb disabled'
)
with open('train_supervised.py', 'w') as f:
    f.write(content)

print("✅ Patches applied!")

✅ Patches applied!


In [18]:
import shutil
shutil.copytree(
    '/content/drive/MyDrive/nsf_results/celeba_erm_seed1',
    '/content/nsf_debiasing/logs/celeba/erm_seed1',
    dirs_exist_ok=True
)
print("✅ Checkpoint restored!")
!ls /content/nsf_debiasing/logs/celeba/erm_seed1/

✅ Checkpoint restored!
args.json
best_checkpoint.pt
checkpoint_0.pt
checkpoint_1.pt
checkpoint_2.pt
checkpoint_3.pt
checkpoint_4.pt
command.sh
events.out.tfevents.1774687233.56effdf89b3a.5374.0
events.out.tfevents.1774687266.56effdf89b3a.6358.0
events.out.tfevents.1774687736.56effdf89b3a.8504.0
events.out.tfevents.1774688023.56effdf89b3a.9950.0
events.out.tfevents.1774688639.56effdf89b3a.12705.0
final_checkpoint.pt
log.txt
resumable_checkpoint.pt
wandb


In [19]:
!python3 ssc.py celeba

2026-03-28 12:11:24.657525: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774699884.680167   62794 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774699884.687748   62794 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774699884.707990   62794 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774699884.708043   62794 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774699884.708049   62794 computation_placer.cc:177] computation placer alr